# 02 - Training and MLflow
Trains Logistic Regression and Random Forest with grid search and 5-fold CV. Takes about a minute. The first cell gets the data itself if it is missing.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working folder:", os.getcwd())
from src.data import CLEAN, prepare
if not CLEAN.exists():
    print("no processed data yet, downloading and cleaning it now")
    prepare()
print("data ready:", CLEAN)

In [ ]:
import json
import pandas as pd
import mlflow
from src import train

## Models and search grids

In [ ]:
for name, (clf, grid) in train.MODELS.items():
    print(name, grid)

## Train (this logs to MLflow in mlflow.db and saves models/model.joblib)

In [ ]:
train.main()

## Runs logged in MLflow

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["heart-disease"], order_by=["start_time DESC"])
cols = ["tags.mlflow.runName", "params.model", "metrics.cv_accuracy", "metrics.cv_precision",
        "metrics.cv_recall", "metrics.cv_roc_auc", "metrics.test_roc_auc", "metrics.loso_roc_auc_mean"]
runs[[c for c in cols if c in runs.columns]].round(3)

## Saved model summary

In [ ]:
meta = json.load(open("models/model_meta.json"))
print("best model:", meta["model"])
print("trained on rows:", meta["rows"])
print({k: v for k, v in meta["metrics"].items() if k.startswith(("cv_", "test_roc", "loso_roc_auc_mean"))})

## Per-hospital test and leave-one-hospital-out AUC (best model)

In [ ]:
m = meta["metrics"]
pd.DataFrame({
    "test_auc": {k.split("_")[-1]: v for k, v in m.items() if k.startswith("test_roc_auc_")},
    "leave_one_out_auc": {k.split("_")[-1]: v for k, v in m.items()
                          if k.startswith("loso_roc_auc_") and not k.endswith("mean")},
})

To see the MLflow UI, run in a terminal from the project folder:
`mlflow ui --backend-store-uri sqlite:///mlflow.db` then open http://127.0.0.1:5000